# 16 · Are these two records the same entity?

## The problem

CRMs, supplier lists and product catalogs fill up with duplicates: *"ACME Corp."* vs *"Acme Corporation Inc"*,
*"Jon Smith"* vs *"Jonathan Smith"*, *"Dr. Maria Garcia"* vs *"María García López"*. String similarity can't tell those
apart from look-alikes such as *"Apple Inc."* vs *"Apple Records"*, or *"Delta Air Lines"* vs *"Delta Faucet"*.

And the two mistakes are **not equal**. A missed merge leaves a duplicate. A **wrong merge** fuses two customers'
histories, invoices and contact details, and that bad data spreads to every downstream system. It's much harder to undo.

## Why Jev

Most pairs don't need a model at all: identical email means same; different country means different. The pairs left
over are judgment calls about **meaning** (abbreviations, nicknames, accents, a company vs. a different company with
the same word in its name), and a big dedup job has thousands of them. Jev is a **System One** model: one calibrated
yes/no probability per pair in about 100 ms, for a fraction of a cent. Calibration is what lets us draw a strict
**merge** line and send the middle to a human.

## What we'll build

```
record A + record B ──► exact rules (free, certain) ── decided? ──► MERGE / SEPARATE
                              │ not decided: the gray zone
                              ▼
                        Jev: P(same entity) ──► p ≥ 0.90 → MERGE
                                                p ≤ 0.20 → SEPARATE
                                                otherwise → REVIEW (a human decides)
```

## What you'll learn
- **Deterministic rules first, Jev for the gray zone**: cheaper, faster, and the easy cases are exactly right.
- How a `Noul` with explicit `NoulCriteria` defines "same entity" (typos and nicknames count, look-alikes don't).
- Asymmetric thresholds, and measuring the metric that matters: **false merges**.

*From the use-case catalog: [18 practical Jev use cases for AI agents](https://huggingface.co/blog/karmen-beatapi/18-practical-jev-use-cases-for-ai-agents), #9.*

> **How to run this notebook**
> 1. Pick the **`.venv`** kernel (top right in VS Code, *Kernel → Change kernel* in Jupyter).
> 2. Run the cells **top to bottom** (or *Run All*). Every cell builds on the ones above it.
> 3. If you see `NameError: name '...' is not defined`, the kernel was restarted (switching kernels does this) and forgot the earlier cells. Click **Run All** or *Run All Above*, then continue.

## Step 0 · Setup

This cell:
- loads the API key (`TYPESAFE_API_KEY`) from the `.env` file at the repo root,
- creates the Jev client, `jev`, that every later cell uses,
- imports the Jev question types (here we only need `Noul` and `NoulCriteria`),
- defines `bar()`, a tiny helper that draws a 0..1 probability as a text bar so you can *see* the numbers.

In [1]:
import os, re
from pathlib import Path
from dotenv import load_dotenv
from typesafe_sdk import TypeSafeClient, Noul, NoulCriteria, Score, Choice

# Look for a .env file in this folder or any parent folder
for folder in [Path.cwd(), *Path.cwd().parents]:
    if (folder / ".env").exists():
        load_dotenv(folder / ".env")
        break

jev = TypeSafeClient()   # the judge: fast, cheap, typed answers


def bar(p, width=20):
    """A unicode progress bar for a 0..1 value."""
    filled = round(max(0.0, min(1.0, p)) * width)
    return "█" * filled + "·" * (width - filled)


print("Jev client ready")

Jev client ready


## Step 1 · Deterministic rules

Before calling any model, apply the rules you can trust completely:
- **identical email** (after trimming spaces and lower-casing) → **MERGE**. An email is a near-unique identifier.
- **different country** → **SEPARATE**. This is a *blocking rule*: in a real dedup job it also means you never even
  compare records across countries, which cuts the number of pairs enormously.

`exact_rules()` returns `(decision, reason)`, or `(None, None)` when the rules can't decide and the pair goes to Jev.
The demo lines run the rules on two tiny examples. No API call yet.

In [2]:
def normalize_email(e):
    return (e or "").strip().lower()


def exact_rules(a, b):
    if a.get("email") and normalize_email(a["email"]) == normalize_email(b.get("email")):
        return "MERGE", "identical email"
    if a.get("country") and b.get("country") and a["country"] != b["country"]:
        return "SEPARATE", "different country (blocking rule)"
    return None, None


print(exact_rules({"email": "jsmith@gmail.com"}, {"email": " JSmith@gmail.com "}))
print(exact_rules({"name": "Apple Inc.", "country": "US"}, {"name": "Apple Records", "country": "UK"}))
print(exact_rules({"name": "ACME Corp.", "country": "US"}, {"name": "Acme Corporation Inc", "country": "US"}))

('MERGE', 'identical email')
('SEPARATE', 'different country (blocking rule)')
(None, None)


**What to notice**
- The email rule ignores case and stray spaces: same email, **MERGE**.
- Different country: **SEPARATE**, without any model call.
- ACME vs Acme Corporation: the rules **can't decide** (`None`). That's the gray zone, and it goes to Jev.

## Step 2 · The Jev question for the gray zone

**How a `Noul` works:** `.noul` is P(yes), from 0 to 1. `NoulCriteria(true=..., false=...)` pins down exactly what
*yes* and *no* mean. Here *yes* explicitly **allows abbreviations, typos, nicknames and formatting**, and *no* means
**different entities that only look similar**. Without those definitions, "same entity" is vague.

The `state` will be a dict `{"record_a": {...}, "record_b": {...}}`: whole records as nested dicts, and the
instruction refers to them as `` `record_a` `` and `` `record_b` ``.

The two thresholds are **asymmetric** on purpose:

| P(same) | decision | why |
|---|---|---|
| `≥ 0.90` (`MERGE_AT`) | **MERGE** | a wrong merge is expensive, so we need to be very sure |
| `≤ 0.20` (`SEPARATE_AT`) | **SEPARATE** | a missed duplicate is cheap, so "probably different" is enough |
| in between | **REVIEW** | a human decides |

This cell only **defines** the question and thresholds.

In [3]:
MATCH_QUESTION = {"same": Noul(
    instructions="Do `record_a` and `record_b` describe the same real-world entity?",
    criteria=NoulCriteria(
        true="Same person or company, allowing for abbreviations, typos, nicknames and formatting",
        false="Different entities that only look similar",
    ),
)}

MERGE_AT, SEPARATE_AT = 0.90, 0.20
print(f"merge at P(same) >= {MERGE_AT}, separate at <= {SEPARATE_AT}, review in between")

merge at P(same) >= 0.9, separate at <= 0.2, review in between


## Step 3 · The matcher: rules first, then Jev

`match()` runs the exact rules; only if they can't decide does it call Jev and apply the thresholds. `why` records the
reason, so every merge decision is explainable later. The guard at the top gives a clear message if the kernel
restarted. This cell only defines the function.

In [4]:
def match(a, b, client):
    if "MATCH_QUESTION" not in globals():
        raise RuntimeError("MATCH_QUESTION is missing: the kernel restarted. Use 'Run All' (or run the cells above) first.")

    decision, why = exact_rules(a, b)
    if decision:
        return {"decision": decision, "why": why, "p": None}
    p = client.system_one(state={"record_a": a, "record_b": b}, questions=MATCH_QUESTION).nouls["same"].noul
    if p >= MERGE_AT:
        decision = "MERGE"
    elif p <= SEPARATE_AT:
        decision = "SEPARATE"
    else:
        decision = "REVIEW"
    return {"decision": decision, "why": f"jev P(same)={p:.2f}", "p": p}


print("match() defined")

match() defined


## Step 4 · Run on labelled pairs and count the mistakes

Six pairs, each with a hand-written label (`True` = really the same entity): two easy rule cases, two look-alike
companies, and two genuine duplicates with different formatting. For each pair we print ✓ if the decision is safe
(correct, or sent to REVIEW) and ✗ if it's wrong. Then we count:
- **false merges**: different entities that we merged. The number to drive to zero.
- **missed duplicates**: same entity that we separated. Cheaper, but still worth tracking.
- **sent to review**: the human workload.

In [5]:
PAIRS = [
    ({"name": "ACME Corp.", "city": "Austin", "country": "US", "phone": "512-555-0101"},
     {"name": "Acme Corporation Inc", "city": "Austin, TX", "country": "US", "phone": "(512) 555 0101"}, True),
    ({"name": "Jon Smith", "email": "jsmith@gmail.com", "country": "US"},
     {"name": "Jonathan Smith", "email": "JSmith@gmail.com ", "country": "US"}, True),
    ({"name": "Apple Inc.", "city": "Cupertino", "country": "US", "industry": "consumer electronics"},
     {"name": "Apple Records", "city": "London", "country": "UK", "industry": "music label"}, False),
    ({"name": "Apple Inc.", "city": "Cupertino", "country": "US", "industry": "consumer electronics"},
     {"name": "Apple Records Inc.", "city": "New York", "country": "US", "industry": "music label"}, False),
    ({"name": "Dr. Maria Garcia", "city": "Madrid", "country": "ES", "specialty": "cardiology"},
     {"name": "María García López", "city": "Madrid", "country": "ES", "specialty": "cardiologist"}, True),
    ({"name": "Delta Air Lines", "city": "Atlanta", "country": "US"},
     {"name": "Delta Faucet Company", "city": "Indianapolis", "country": "US"}, False),
]
false_merges = missed = reviews = 0
for a, b, same in PAIRS:
    d = match(a, b, jev)
    false_merges += d["decision"] == "MERGE" and not same
    missed += d["decision"] == "SEPARATE" and same
    reviews += d["decision"] == "REVIEW"
    ok = "✓" if (d["decision"] == "MERGE") == same or d["decision"] == "REVIEW" else "✗"
    print(f"{ok} [{d['decision']:^8}] {a['name']:<18} vs {b['name']:<22} {d['why']}")
print(f"\nfalse merges: {false_merges}  (the number to drive to zero)   missed duplicates: {missed}   sent to review: {reviews}")

✓ [ MERGE  ] ACME Corp.         vs Acme Corporation Inc   jev P(same)=0.95
✓ [ MERGE  ] Jon Smith          vs Jonathan Smith         identical email
✓ [SEPARATE] Apple Inc.         vs Apple Records          different country (blocking rule)
✓ [SEPARATE] Apple Inc.         vs Apple Records Inc.     jev P(same)=0.04


✓ [ REVIEW ] Dr. Maria Garcia   vs María García López     jev P(same)=0.82
✓ [SEPARATE] Delta Air Lines    vs Delta Faucet Company   jev P(same)=0.02

false merges: 0  (the number to drive to zero)   missed duplicates: 0   sent to review: 1


**What to notice**
- Two of the six pairs never reached Jev: Jon Smith (identical email) and Apple vs Apple Records **London** (different
  country). Free and exactly right.
- ACME vs Acme Corporation: same city, same phone in a different format. Jev scores it above 0.90: **MERGE**.
- Apple Inc. vs Apple Records **Inc.** (both US) and Delta Air Lines vs Delta Faucet: shared words, different
  industries. Jev scores both **near 0**: **SEPARATE**. String similarity would have been tempted.
- Dr. Maria Garcia vs María García López is very likely the same cardiologist, and Jev agrees (around 0.8), but that is
  **below the strict 0.90 merge line**, so it goes to **REVIEW**. That's the asymmetric threshold doing its job: "very
  likely" isn't enough to risk a wrong merge.
- **Zero false merges** on this set, with one pair for a human to check.

## Step 5 · Give it a face

Everything from here on is **presentation only**. This cell is a small style kit (colours, `box`, `badge`, `meter`) and
`form_app()`, which wires text boxes and a button to any function that returns HTML. `match_html()` shows a decision as a card with both records side by side.
**The decision logic above is untouched**: the UI only calls the same functions you've already seen run.

In [6]:
import ipywidgets as widgets
from IPython.display import HTML, display

INK, SLATE, CORAL, SAGE, AMBER, RED, LINE = "#1F1E1D", "#5E6B7A", "#D97757", "#1F7A3D", "#C98A00", "#C0392B", "#DEDACE"
CARD = f"background:#FAF9F5;border:1px solid {LINE};border-radius:10px;padding:14px 16px;margin:8px 0;"
BADGE = "display:inline-block;padding:2px 10px;border-radius:12px;color:white;font-weight:600;font-size:12px;"


def box(inner):
    return f'<div style="font-family:-apple-system,Segoe UI,sans-serif;color:{INK};max-width:860px;">{inner}</div>'


def badge(text, color):
    return f'<span style="{BADGE}background:{color};">{text}</span>'


def meter(label, p, width=160):
    """A labelled horizontal probability bar."""
    col = RED if p > 0.7 else AMBER if p > 0.4 else SAGE
    return (f'<div style="display:flex;align-items:center;gap:8px;font-size:12px;margin:2px 0;">'
            f'<span style="width:190px;color:{SLATE};">{label}</span>'
            f'<span style="width:{width}px;height:8px;background:#ECE9E0;border-radius:4px;overflow:hidden;">'
            f'<span style="display:block;height:8px;width:{p * 100:.0f}%;background:{col};"></span></span>'
            f'<b>{p:.2f}</b></div>')


def form_app(fields, render, button="Run"):
    """fields: list of (name, label, default, rows). render(**values) returns HTML."""
    boxes = {}
    for name, label, default, rows in fields:
        w = (widgets.Textarea(value=default, rows=rows, layout=widgets.Layout(width="90%"))
             if rows > 1 else widgets.Text(value=default, layout=widgets.Layout(width="90%")))
        boxes[name] = w
    btn = widgets.Button(description=button, button_style="warning")
    out = widgets.Output()

    def go(_):
        with out:
            out.clear_output()
            display(HTML(box("<i>thinking...</i>")))
            html = render(**{k: w.value.strip() for k, w in boxes.items()})
            out.clear_output(wait=True)
            display(HTML(html))

    btn.on_click(go)
    rows = [widgets.VBox([widgets.HTML(f"<b>{label}</b>"), boxes[name]]) for name, label, _, _ in fields]
    display(widgets.VBox(rows + [btn]), out)

The next cell defines `match_html()` and renders the first four pairs from Step 4 as cards. It accepts records as
dicts or as JSON strings, which is what the try-it form below sends.

In [7]:
import json

M_COLOR = {"MERGE": SAGE, "REVIEW": AMBER, "SEPARATE": SLATE}


def record_card(rec):
    return "<br>".join(f'<span style="color:{SLATE};">{k}:</span> {v}' for k, v in rec.items())


def match_html(a, b):
    a, b = (json.loads(x) if isinstance(x, str) else x for x in (a, b))
    d = match(a, b, jev)
    col = M_COLOR[d["decision"]]
    return box(f'<div style="{CARD}border-left:5px solid {col};">{badge(d["decision"], col)} '
               f'<span style="color:{SLATE};font-size:12px;">{d["why"]}</span>'
               f'<div style="display:flex;gap:12px;margin-top:8px;font-size:13px;">'
               f'<div style="flex:1;{CARD}">{record_card(a)}</div><div style="flex:1;{CARD}">{record_card(b)}</div></div></div>')


display(HTML(box("".join(match_html(a, b) for a, b, _ in PAIRS[:4]))))

## Step 6 · Try it yourself

Edit the two records (as JSON) and press **Match** (needs a live kernel). Ideas:
- The default, *Intl Business Machines* vs *IBM* in Armonk → an abbreviation Jev knows: **MERGE**.
- `{"name": "Bob's Pizza", "city": "Boston", "country": "US", "address": "12 Main St"}` vs
  `{"name": "Bobs Pizza", "city": "Boston", "country": "US", "address": "12 Main Street"}` → same address, formatting
  differences: **MERGE**.
- Change the second one to `{"name": "Bobs Pizzeria", "city": "Boston", "country": "US", "phone": "617-555-0142"}` and
  give the first a different phone → Jev is unsure (around the middle): **REVIEW**.
- `{"name": "John Smith", "city": "Chicago", "country": "US"}` vs `{"name": "John Smith", "city": "Denver", "country": "US"}`
  → identical names are **not** enough: **REVIEW**, leaning towards different.

In [8]:
form_app([("a", "Record A (JSON)", '{"name": "Intl Business Machines", "city": "Armonk", "country": "US"}', 2),
          ("b", "Record B (JSON)", '{"name": "IBM", "city": "Armonk, New York", "country": "US"}', 2)],
         match_html, "Match")

Output()

## Recap and exercises

**Key ideas**
1. **Rules first, Jev for the gray zone.** Exact rules are free and certain; only the ambiguous pairs need a model.
2. **`NoulCriteria` defines the question.** Spelling out that nicknames and formatting count, and look-alikes don't, is
   what makes "same entity" answerable.
3. **Asymmetric thresholds for asymmetric costs.** Merge only when very sure; separate when probably different; review
   the middle.
4. **Measure the mistake that matters.** For dedup, that's false merges, then review workload.

**Exercises**
- Lower `MERGE_AT` to 0.75 and re-run Step 4. What merges now, and is it right? What would a wrong one cost?
- Shared mailboxes break the email rule: give two *different* companies the email `info@gmail.com` and see what
  happens. Add an exception for generic addresses (`info@`, `sales@`, `admin@`).
- Add a phone-number rule: strip everything but digits, and treat identical phone numbers as strong evidence (but maybe
  not a hard MERGE: think about switchboards).
- Add five labelled pairs from your own data and track false merges, missed duplicates and review count together.